# 03 — Fine-tuning
Fine-tunes TigerLLM-1B-it on a curated bilingual medical dataset using LoRA (PEFT), 
producing a lightweight adapter for Swasthya's offline medical assistant.

**Input:** `combined_data.json` (2,567 EN) + `bengali_samples.json` (500 manually translated BN)  
**Subset used:** 1,000 English + 500 Bengali (1,500 samples total)  
**Output:** LoRA adapter pushed to HuggingFace as `V2-TigerLLM-Medical-BN`

## Environment Setup

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## Installing Dependencies

In [2]:
#install libraries
!pip install -q peft trl bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 12.0 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 85.6 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.

In [3]:
import torch
import transformers
import trl
import peft

print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"TRL          : {trl.__version__}")
print(f"PEFT         : {peft.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch      : 2.10.0+cu128
Transformers : 5.0.0
TRL          : 1.5.1
PEFT         : 0.18.1
CUDA available: True


## Loading Dataset

In [8]:
import json

with open('/kaggle/input/datasets/souravchandra02/swasthya-medical-data/combined_data.json', 
          'r', encoding='utf-8') as f:
    combined_data = json.load(f)

print(f"Loaded: {len(combined_data)} samples")

Loaded: 5134 samples


In [6]:
import json

with open('/kaggle/input/datasets/souravchandra02/swasthya-medical-data-bengali/bengali_samples.json', 
          'r', encoding='utf-8') as f:
    bengali_samples = json.load(f)

print(f"Loaded: {len (bengali_samples)} samples")

Loaded: 508 samples


## Preparing Training Subset

In [9]:
import random
random.seed(42)

english_samples = [s for s in combined_data if s['language'] == 'en']

english_subset = random.sample(english_samples, 1000)
bengali_subset = random.sample(bengali_samples, 500)

subset_data = english_subset + bengali_subset
random.shuffle(subset_data)

print(f"Subset size: {len(subset_data)}")

Subset size: 1500


## System Prompt

In [10]:
#system prompt
SYSTEM_PROMPT = """তুমি স্বাস্থ্যা, একটি স্বাস্থ্য সহায়ক।

গুরুত্বপূর্ণ নিয়ম:

* ব্যবহারকারী যে ভাষায় প্রশ্ন করবে, সেই ভাষাতেই উত্তর দেবে
* ব্যবহারকারী ইংরেজিতে লিখলে ইংরেজিতে উত্তর দাও।
* রোগ নির্ণয় নিশ্চিতভাবে করবে না।
* সাধারণ স্বাস্থ্য পরামর্শ দেবে।
* জরুরি লক্ষণ থাকলে দ্রুত ডাক্তার বা হাসপাতালে যেতে বলবে।

"""

## HuggingFace Authentication

In [11]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

In [12]:
from huggingface_hub import login
login(token=hf_token)

In [13]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

## Loading Base Model & Tokenizer

In [14]:
#model
model_name = "md-nishat-008/TigerLLM-1B-it"

In [15]:
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto"
)

print(f"GPU Memory Used: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"GPU Memory Free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1024**3:.2f} GB")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/197 [00:00<?, ?B/s]

GPU Memory Used: 0.91 GB
GPU Memory Free: 13.65 GB


In [16]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

## Formatting Dataset

In [18]:
# Format dataset

def format_sample(sample):
    messages = [
        {"role": "user", "content": sample["input"]},
        {"role": "assistant", "content": sample["output"]},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )

    return {"text": text}


formatted_data = [format_sample(s) for s in subset_data]


print(f"Total formatted: {len(formatted_data)}")
print("\nSample formatted text:")
print(formatted_data[0]['text'])

Total formatted: 1500

Sample formatted text:
<bos><start_of_turn>user
Hello Doctor, am 20 years old, am having a big problem with my skin, am a dark girl but my thighs are full of dry patches, pigments i would say, that turn to white when sratched.They dont itch in any way but make me feel uncomfortable. They normally tend to reduce when i use an anti-dry cream but eventually come back when usage is stopped. Please help me out in anyway and give me a solution that will help me get rid of them entirely so i can have a smooth skin thruoughout. My other body parts are very smooth with no pigments, no acne.Thank you very much<end_of_turn>
<start_of_turn>model
Hello, Do you have a family history of similar lesions? From your description, it sounds like severe The medical term used is necrosis. You have to continue using moisturizers and body butters. Apply them within 3 minutes after a shower on damp skin to seal in the moisture. You could consult a dermatologist who could rule out eczema.

## Train / Validation Split

In [19]:
from datasets import Dataset
import pandas as pd

df=pd.DataFrame(formatted_data)
dataset = Dataset.from_pandas(df)

dataset=dataset.train_test_split(test_size=0.1, seed=42)

print(f"Train samples    : {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['test'])}")
print(f"\nColumn names: {dataset['train'].column_names}")

Train samples    : 1350
Validation samples: 150

Column names: ['text']


## Baseline Evaluation (Before Fine-tuning)

In [20]:
# Baseline evaluation — before fine-tuning
def generate_response(prompt, max_new_tokens=200):

    messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to("cuda")

    input_length = inputs.input_ids.shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    )

    return response.strip()

In [21]:
# Test prompts — English and Bengali
test_prompts = [
    # English — common symptoms
    "I have fever since 2 days and headache. What should I do?",
    "My child has cough and cold since 3 days and is not eating.",
    
    # Bengali — common symptoms  
    "আমার দুই দিন ধরে জ্বর এবং মাথাব্যথা। আমার কী করা উচিত?",
    "আমার পেটে ব্যথা হচ্ছে এবং বমি বমি ভাব লাগছে।",
    
    # English — mental health
    "I feel very anxious and cannot sleep at night.",
    
    # Bengali — mental health
    "কয়েক সপ্তাহ ধরে আমার খুব দুশ্চিন্তা হচ্ছে এবং রাতে ঘুম হচ্ছে না।",
    
    # English — minor injury
    "I burned my hand while cooking. What should I do?",
    
    # Bengali — child health
    "আমার ৫ বছরের বাচ্চার জ্বর আছে এবং সে কিছু খাচ্ছে না।",
    
    # English — skin issue
    "I have a rash on my arms that is very itchy since yesterday.",
    
    # Bengali — dizziness
    "আজ সকাল থেকে বারবার মাথা ঘুরছে এবং দাঁড়ালে ভারসাম্য রাখতে কষ্ট হচ্ছে।",
    
    # Emergency — should redirect immediately
    "I have severe chest pain spreading to my left arm.",
    
    # Emergency Bengali
    "আমার হঠাৎ বুকে তীব্র ব্যথা হচ্ছে এবং বাম হাতে ছড়িয়ে যাচ্ছে।"
]

In [22]:
# Generate and save baseline responses
baseline_results = []

for prompt in test_prompts:
    response = generate_response(prompt)
    baseline_results.append({
        "prompt": prompt,
        "response": response
    })
    print(f"PROMPT: {prompt}")
    print(f"RESPONSE: {response}")
    print()

PROMPT: I have fever since 2 days and headache. What should I do?
RESPONSE: It sounds like you might be experiencing a fever and headache. It's important to get this checked out by a doctor or healthcare professional as soon as possible, especially if the symptoms are persistent or worsening. You can also try some home remedies such as drinking plenty of fluids, resting, and taking over-the-counter medications for your fever and headache. However, it's always best to seek medical advice for any health concerns.

PROMPT: My child has cough and cold since 3 days and is not eating.
RESPONSE: I'm sorry to hear that your child has been coughing and having a cold for three days. It's important to monitor their symptoms closely and seek medical attention if they are experiencing any concerning symptoms such as fever, difficulty breathing, or severe pain. If you have any concerns about your child's health, please contact a doctor or healthcare professional immediately.

PROMPT: আমার দুই দিন ধর

In [23]:
# Save baseline to Drive
import json

with open('/kaggle/working/baseline_results.json',
          'w', encoding='utf-8') as f:
    json.dump(baseline_results, f, ensure_ascii=False, indent=2)

print("Baseline results saved!")

Baseline results saved!


## LoRA Configuration

In [24]:
from peft import LoraConfig, get_peft_model


In [28]:
# LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

## Training Configuration

In [29]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir="/kaggle/working/checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=30,
    learning_rate=5e-5,
    bf16=True,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=85,
    save_strategy="steps",
    save_steps=50, 
    load_best_model_at_end=False,
    report_to="none",
    optim="adamw_torch",
    gradient_checkpointing=False
)

print("SFTConfig configured!")

SFTConfig configured!


In [30]:
steps_per_epoch = len(dataset['train']) // (2 * 4)
total_steps = steps_per_epoch * 2

print(f"Samples per epoch : {len(dataset['train'])}")
print(f"Steps per epoch   : {steps_per_epoch}")
print(f"Total steps       : {total_steps}")
print(f"Checkpoints saved every 200 steps to Drive")

Samples per epoch : 1350
Steps per epoch   : 168
Total steps       : 336
Checkpoints saved every 200 steps to Drive


## Initializing Trainer

In [31]:
from trl import SFTTrainer
from transformers import DataCollatorForLanguageModeling

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    peft_config=lora_config,
    args=sft_config,
    processing_class=tokenizer,
)

print("Trainer configured!")

Adding EOS to train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Trainer configured!


## Training

In [ ]:
#start training
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 106}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
85,3.505406,3.337266,3.233769,123584.000000,0.395715
170,3.221294,3.145567,3.075882,250098.000000,0.413219
255,3.181296,3.102891,3.056479,377161.000000,0.416154


In [33]:
trainer.train(resume_from_checkpoint="/kaggle/working/checkpoints/checkpoint-250")

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
255,3.181296,3.103527,3.054773,504790.000000,0.416722


TrainOutput(global_step=338, training_loss=0.8125198296541293, metrics={'train_runtime': 354.269, 'train_samples_per_second': 7.621, 'train_steps_per_second': 0.954, 'total_flos': 2383586457060864.0, 'train_loss': 0.8125198296541293, 'entropy': 2.9840551731633207, 'num_tokens': 625185.0, 'mean_token_accuracy': 0.43285104048018364, 'epoch': 2.0})

In [34]:
model = trainer.model
model.config.use_cache = True

## Post-training Evaluation (After Fine-tuning)

In [35]:
def generate_response(prompt, max_new_tokens=200):

    messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to("cuda")

    input_length = inputs.input_ids.shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    )

    return response.strip()

In [36]:
# Same test prompts as baseline
test_prompts = [
    # English — common symptoms
    "I have fever since 2 days and headache. What should I do?",
    "My child has cough and cold since 3 days and is not eating.",
    
    # Bengali — common symptoms  
    "আমার দুই দিন ধরে জ্বর এবং মাথাব্যথা। আমার কী করা উচিত?",
    "আমার পেটে ব্যথা হচ্ছে এবং বমি বমি ভাব লাগছে।",
    
    # English — mental health
    "I feel very anxious and cannot sleep at night.",
    
    # Bengali — mental health
    "কয়েক সপ্তাহ ধরে আমার খুব দুশ্চিন্তা হচ্ছে এবং রাতে ঘুম হচ্ছে না।",
    
    # English — minor injury
    "I burned my hand while cooking. What should I do?",
    
    # Bengali — child health
    "আমার ৫ বছরের বাচ্চার জ্বর আছে এবং সে কিছু খাচ্ছে না।",
    
    # English — skin issue
    "I have a rash on my arms that is very itchy since yesterday.",
    
    # Bengali — dizziness
    "আজ সকাল থেকে বারবার মাথা ঘুরছে এবং দাঁড়ালে ভারসাম্য রাখতে কষ্ট হচ্ছে।",
    
    # Emergency — should redirect immediately
    "I have severe chest pain spreading to my left arm.",
    
    # Emergency Bengali
    "আমার হঠাৎ বুকে তীব্র ব্যথা হচ্ছে এবং বাম হাতে ছড়িয়ে যাচ্ছে।"
]

finetuned_results = []

for prompt in test_prompts:
    response = generate_response(prompt)
    finetuned_results.append({
        "prompt": prompt,
        "response": response
    })
    print(f"PROMPT: {prompt}")
    print(f"RESPONSE: {response}")
    print()
    

PROMPT: I have fever since 2 days and headache. What should I do?
RESPONSE: Hello, I understand your concern. You are having a fever since two days and headache. It is possible that you have a viral infection or some other illness. If you have any other symptoms such as body aches, fatigue, sore throat, cough, etc., please let me know. I will try to help you in the best way. Thanks for your query.

PROMPT: My child has cough and cold since 3 days and is not eating.
RESPONSE: Hi, I understand your concern. Cough and cold are common symptoms of a viral infection. It is possible that your child has a respiratory virus. You can try some home remedies like giving him warm tea with honey or lemon juice, and rest. If the symptoms persist for more than 10 days, you should consult a doctor. Hope this helps!

PROMPT: আমার দুই দিন ধরে জ্বর এবং মাথাব্যথা। আমার কী করা উচিত?
RESPONSE: হ্যালো, আপনার উপসর্গগুলো খুবই সাধারণ। সাধারণত জ্বর ও মাথাব্যথার কারণ হতে পারে ঠান্ডা লাগা, বায়ুচাপের পরিবর্তন, ঘুমে

## Saving & Pushing to HuggingFace

In [39]:
save_path = "/kaggle/working/v2-tigerllm-medical-lora"

trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Saved to {save_path}")

Saved to /kaggle/working/v2-tigerllm-medical-lora


In [40]:
import shutil

shutil.make_archive(
    "/kaggle/working/v2-tigerllm_medical_lora",
    "zip",
    save_path
)

print("ZIP created!")

ZIP created!


In [41]:
from peft import PeftModel

merged_model = trainer.model.merge_and_unload()

merged_model.save_pretrained(
    "/kaggle/working/v2-tigerllm-medical-merged",
    safe_serialization=True
)

tokenizer.save_pretrained(
    "/kaggle/working/v2-tigerllm-medical-merged"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/v2-tigerllm-medical-merged/tokenizer_config.json',
 '/kaggle/working/v2-tigerllm-medical-merged/chat_template.jinja',
 '/kaggle/working/v2-tigerllm-medical-merged/tokenizer.json')

In [42]:
from huggingface_hub import login

login(hf_token)

merged_model.push_to_hub(
    "souravchandra01/V2-TigerLLM-Medical-BN"
)

tokenizer.push_to_hub(
    "souravchandra01/V2-TigerLLM-Medical-BN"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/souravchandra01/V2-TigerLLM-Medical-BN/commit/8f0f2fd21e6528f8c3597a3133aee6b73a8bf0e7', commit_message='Upload tokenizer', commit_description='', oid='8f0f2fd21e6528f8c3597a3133aee6b73a8bf0e7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/souravchandra01/V2-TigerLLM-Medical-BN', endpoint='https://huggingface.co', repo_type='model', repo_id='souravchandra01/V2-TigerLLM-Medical-BN'), pr_revision=None, pr_num=None)